In [1]:
import doranet.modules.enzymatic as enzymatic
import doranet.modules.synthetic as synthetic
import doranet.modules.post_processing as post_processing

[09:28:30] WARNING: not removing hydrogen atom without neighbors
[09:28:30] WARNING: not removing hydrogen atom without neighbors
[09:28:30] WARNING: not removing hydrogen atom without neighbors
[09:28:30] WARNING: not removing hydrogen atom without neighbors
[09:28:30] WARNING: not removing hydrogen atom without neighbors


In [11]:
from rdkit import Chem

basidalin_smiles = "NC1=CC(=O)O/C1=C/C=O"

mol = Chem.MolFromSmiles(basidalin_smiles)
if mol is None:
    raise ValueError(f"Invalid SMILES: {basidalin_smiles}")

canonical = Chem.MolToSmiles(mol, canonical=True)
canonical_isomeric = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

print("Input SMILES:          ", basidalin_smiles)
print("Canonical SMILES:      ", canonical)
print("Canonical (isomeric):  ", canonical_isomeric)

Input SMILES:           NC1=CC(=O)O/C1=C/C=O
Canonical SMILES:       NC1=CC(=O)O/C1=C/C=O
Canonical (isomeric):   NC1=CC(=O)O/C1=C/C=O


In [12]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

def smilesSimilarity(smilesA: str, smilesB: str, radius: int = 2, nBits: int = 2048) -> float:
    molA = Chem.MolFromSmiles(smilesA)
    molB = Chem.MolFromSmiles(smilesB)

    if molA is None:
        raise ValueError(f"Invalid SMILES A: {smilesA}")
    if molB is None:
        raise ValueError(f"Invalid SMILES B: {smilesB}")

    fpA = AllChem.GetMorganFingerprintAsBitVect(molA, radius, nBits=nBits)
    fpB = AllChem.GetMorganFingerprintAsBitVect(molB, radius, nBits=nBits)

    return DataStructs.TanimotoSimilarity(fpA, fpB)

# Example usage
smiles1 = "NC1=CC(=O)O/C1=C/C=O"   
smiles2 = "C1=C(C(=CC=O)OC1=O)N"   

score = smilesSimilarity(smiles1, smiles2)
print("Tanimoto similarity:", round(score, 3))

# Simple decision rule (tune threshold to your use-case)
threshold = 0.7
print("Similar" , score >= threshold)

Tanimoto similarity: 1.0
Similar True


[10:01:20] DEPRECATION WARNING: please use MorganGenerator
[10:01:20] DEPRECATION WARNING: please use MorganGenerator


In [2]:
# copy in SMILES string of the molecule you want to modify
user_starters = {'C1=C(C(=CC=O)OC1=O)N'}

# I typically use these for synthetic chemistry reactions
# for enzymatic reactions, the cofactor/ helper molecules needed are already in DORAnet by default
user_helpers = {'O','O=O','[H][H]','O=C=O','C=O','[C-]#[O+]','Br','[Br][Br]','CO',
                'C=C','O=S(O)O','N','O=S(=O)(O)O','O=NO','N#N','O=[N+]([O-])O','NO',
                'C#N','S','O=S=O','N#CO'}

# copy in SMILES string of the molecule you want to reach
# if you are only modifying the starting molecule, you don't need to specify a target
user_target = {'OC1=CC=CC=C1'}

In [3]:
job_name = "test"

#### Performing enzymatic modifications only

In [4]:
enzymatic_forward_network = enzymatic.generate_network(
    job_name = job_name,
    starters = user_starters,
    gen = 1, # number of generations/ steps
    max_atoms = {'C':5}, # you can create a dictionary to limit maximum atoms
    direction = "forward")

Job name: test
Job type: enzymatic network expansion forward
Job started on: 2026-01-21 09:28:41.501322


[09:28:41] WARNING: not removing hydrogen atom without neighbors
[09:28:41] WARNING: not removing hydrogen atom without neighbors
[09:28:54] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 1 O, 3, is greater than permitted
[09:28:54] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 1 O, 3, is greater than permitted
[09:28:54] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:28:54] Explicit valence for atom # 0 C, 5, is greater than perm

Number of generations: 1
Number of operators loaded: 3571
Number of molecules before expantion (including cofactors): 42
Number of molecules after expantion (including cofactors): 42
Number of reactions: 0
Time used for network generation: 0.21 minutes



In [6]:
# print out the SMILES string of all molecules produced in the forward network
for mol in enzymatic_forward_network.mols:
    print(mol.uid)


Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O)OP(=O)(O)O)[C@@H](O)[C@H]1O
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C@@H](O)[C@H]1O
Cc1cc2nc3c(=O)[nH]c(=O)nc-3n(C[C@H](O)[C@H](O)[C@H](O)COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](O)[C@@H]3O)c2cc1C
Cc1cc2c(cc1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O)(O)OP(=O)(O)OC[C@H]1O[C@@H](n3cnc4c(N)ncnc43)[C@H](O)[C@@H]1O)c1[nH]c(=O)[nH]c(=O)c1N2
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OP(=O)(O)O)[C@@H](O)[C@H]1O
NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)c1
NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)OC[C@H]3O[C@@H](n4cnc5c(N)ncnc54)[C@H](OP(=O)(O)O)[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)OS(=O)(=O)O)[C@@H](OP(=O)(O)O)[C@H]1O
Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)(O)O)[C@@H](OP(=O)(O)O)[C@H]1O
C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(N)ncnc32)[C@H](O)[C@@H]1O
Nc1ncnc2c1ncn2[C@@H]1O[C@H](CSCC[C@H](N)C(=O)O)[C@@H](O)[C@H]1O
O=c1c

#### Performing synthetic chemistry modifications only

In [7]:
synthetic_forward_network = synthetic.generate_network(
    job_name = job_name,
    starters = user_starters,
    helpers = user_helpers,
    gen = 1,
    direction = "forward"
)

Job name: test
Job type: synthetic network expansion forward
Job started on: 2026-01-21 09:33:30.988549


[09:33:31] product 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 2 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] reactant 1 has no mapped atoms.
[09:33:31] product atom-mapping number 5 found multiple times.
[09:33:31] reactant 2 has no mapped atoms.
[09:33:31] product atom-mapping number 3 found multiple times.
[09:33:31] product atom-mapping number 1 found multiple times.
[09:33:31] product atom-mapping number 2 found multiple times.
[09:33:31] product atom-mapping number 4 found multiple times.
[09:33:31] reactant 2 has no mapped atoms.
[09:33:31] reactant 2 has no mapped atoms.
[09:33:31] product atom-mapping number 5 

Number of generations: 1
Number of operators: 386
Number of molecules before expantion: 22
Number of molecules after expantion: 291
Number of reactions: 287
Time used for network generation: 0.05 minutes



In [8]:
# print out the SMILES string of all molecules produced in the synthetic chemistry
for mol in synthetic_forward_network.mols:
    print(mol.uid)

[H][H]
[C-]#[O+]
O=S(=O)(O)O
O=[N+]([O-])O
N
C=C
BrBr
C=O
O=C=O
NO
S
CO
N#CO
N#N
C#N
O=NO
O=S=O
O=O
O=S(O)O
Br
O
NC1=CC(=O)OC1=CC=O
NC1=C2OC(=O)C1C1OC21
NC12OC=CC13OC(=O)C23
NC1(C=C=O)C(=O)C1C=O
NC1=C=C(O)OC1=CC=O
NC1=CC(=O)OC1=C=CO
N=C1CC(=O)OC1=CC=O
C=C1OC(=O)C=C1N
NC1=CC2(O)OC1=C2C=O
NC1=C2C(=O)OC1=CC2O
O=CC=C1OC(=O)C2NC12
O=CC1NC2=CC(=O)OC21
O=CCC12NC1=CC(=O)O2
O=CC1NC2=C1OC(=O)C2
O=CCC1=C2NC2C(=O)O1
O=C1C=C2N=CC=C2O1
O=CC=c1oc2cc1n2
NC1CC(=O)OC1=CC=O
NC1=CC(=O)OC1CC=O
NC1=CC(=O)OC1=CCO
NC1=CCOC1=CC=O
CC=C1OC(=O)C=C1N
NC(=CC=O)C(O)=CC=O
NC(=CCO)C(O)=CC=O
O=C1C=C2NCC=C2O1
NC1=CC(=O)OC(=O)C1=CC=O
NC1(C=O)CC(=O)OC1=CC=O
NC1C(=CC=O)OC(=O)C1C=O
NC1=CC(=O)OC1C(C=O)C=O
NC1=CC(=O)OC1(C=O)CC=O
NC1(OS(=O)(=O)O)CC(=O)OC1=CC=O
NC1C(=CC=O)OC(=O)C1OS(=O)(=O)O
NC1=CC(=O)OC1C(C=O)OS(=O)(=O)O
NC1=CC(=O)OC1(CC=O)OS(=O)(=O)O
NC1(C(=O)OS(=O)(=O)O)CC(=O)OC1=CC=O
NC1C(=CC=O)OC(=O)C1C(=O)OS(=O)(=O)O
NC1=CC(=O)OC1C(C=O)C(=O)OS(=O)(=O)O
NC1=CC(=O)OC1(CC=O)C(=O)OS(=O)(=O)O
NC1(C(=O)OS(=O)(=O)O)CC(=O)OC1C(C=

#### Performing both enzymatic and synthetic chemistry modifications

In [9]:
# create a forward enzymatic network
# and a reverse synthetic network
# then join them in the middle
forward_network = enzymatic.generate_network(
    job_name = job_name,
    starters = user_starters,
    gen = 1,
    direction = "forward")

reverse_network = synthetic.generate_network(
    job_name = job_name,
    starters = user_starters,
    helpers = user_helpers,
    gen = 1,
    direction = "reverse") # or 'retro' if 'reverse' gives you an error

Job name: test
Job type: enzymatic network expansion forward
Job started on: 2026-01-21 09:33:51.626467


[09:33:51] WARNING: not removing hydrogen atom without neighbors
[09:33:51] WARNING: not removing hydrogen atom without neighbors
[09:34:04] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 1 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 1 O, 3, is greater than permitted
[09:34:04] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 1 O, 3, is greater than permitted
[09:34:04] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 0 C, 5, is greater than permitted
[09:34:04] Explicit valence for atom # 0 C, 5, is greater than perm

Number of generations: 1
Number of operators loaded: 3571
Number of molecules before expantion (including cofactors): 42
Number of molecules after expantion (including cofactors): 98
Number of reactions: 59
Time used for network generation: 0.37 minutes

Job name: test
Job type: synthetic network expansion reverse
Job started on: 2026-01-21 09:34:16.205118


UnboundLocalError: local variable 'smarts_list' referenced before assignment

#### Enumerate pathways

In [10]:
post_processing.one_step(
    networks = {
        forward_network,
        },
    total_generations = 1,
    starters = user_starters,
    helpers = user_helpers,
    target = user_target,
    job_name = job_name,
    )

Job name: test
Job type: post-processing pretreat networks
Job started on: 2026-01-21 09:34:41.085630
Loading networks, it may take a while if loading large networks from file
Loading network 1 from memory
Number of reations in network 1: 59
Networks loaded, now generating reaction strings
Reaction strings generation finished
Removing unconnected reactions
Unconnected reactions removed
Total number of reactions after pretreatment: 59
Time used for network pretreatment: 0.00 minutes

Job name: test
Job type: pathway search
Job started on: 2026-01-21 09:34:41.141561
Pathway finder started, total number of reactions in network 59
Searching for pathways.
If it is taking too long, try adjusting pruning parameters
No pathway found! Try adjusting pruning parameters.
Pathway search finished, removing loops if there's any.
Time used for pathway search: 0.00 minutes

Job name: test
Job type: pathway ranking
Job started on: 2026-01-21 09:34:41.162533
Pathway file not found, exiting pathway rankin

[09:34:41] WARNING: not removing hydrogen atom without neighbors
[09:34:41] WARNING: not removing hydrogen atom without neighbors
[09:34:41] WARNING: not removing hydrogen atom without neighbors


In [13]:
import json
from pathlib import Path

path = Path("test_network_pretreated.json")

with path.open("r") as f:
    data = json.load(f)

print("Top-level type:", type(data))
if isinstance(data, dict):
    print("Top-level keys:", sorted(data.keys())[:50])

    # Common network-like fields to probe
    for k in ["mols", "molecules", "nodes", "rxns", "reactions", "edges", "links", "graph"]:
        if k in data:
            v = data[k]
            try:
                print(f"{k}: type={type(v).__name__}, len={len(v)}")
            except TypeError:
                print(f"{k}: type={type(v).__name__}")

elif isinstance(data, list):
    print("List length:", len(data))
    print("First element type:", type(data[0]) if data else None)


Top-level type: <class 'list'>
List length: 59
First element type: <class 'str'>
